In [1]:
import pandas as pd
import numpy as np

from fasttext import train_supervised

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from tqdm.notebook import tqdm

# importing mean() 
from statistics import mean


In [2]:
!unzip -o '/kaggle/input/jigsaw-toxic-comment-classification-challenge/*.zip' -d /kaggle/working > /dev/null



3 archives were successfully processed.


In [3]:
train_text = pd.read_csv("train.csv")
test_text = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print(train_text.shape, test_text.shape, sample_submission.shape)
train_text.head()


(159571, 8) (153164, 2) (153164, 7)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [4]:
y_cols = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]


In [5]:
s = "\n abc"
s.replace("\n"," ")


'  abc'

In [6]:
# Lets do some cleaning of this text
def clean_it(text,normalize=True):
    # Replacing possible issues with data. We can add or reduce the replacemtent in this chain
    s = str(text).replace(',',' ').replace('"','').replace('\'',' \' ').replace('.',' . ').replace('(',' ( ').\
            replace(')',' ) ').replace('!',' ! ').replace('?',' ? ').replace(':',' ').replace(';',' ').lower()
    s = s.replace("\n"," ")
    
    # normalizing / encoding the text
    if normalize:
        s = s.normalize('NFKD').str.encode('ascii','ignore').str.decode('utf-8')
    
    return s

# Now lets define a small function where we can use above cleaning on datasets
def clean_df(data, cleanit= False, shuffleit=False, encodeit=False, label_prefix='__class__'):
    # Defining the new data
    df = data[['comment_text']].copy(deep=True)
    for col in y_cols:
        df[col] = label_prefix + data[col].astype(str) + ' '
    
    # cleaning it
    if cleanit:
        df['comment_text'] = df['comment_text'].apply(lambda x: clean_it(x,encodeit))
    
    # shuffling it
    if shuffleit:
        df.sample(frac=1).reset_index(drop=True)
            
    return df


In [7]:
X = train_text.comment_text
y = train_text[["toxic","severe_toxic","obscene","threat","insult","identity_hate"]]

train, val = train_test_split(train_text, shuffle = True, random_state = 123)

# Transform the datasets using the above clean functions
df_train_cleaned = clean_df(train, True, True)
df_val_cleaned = clean_df(val, True, True, label_prefix='')


In [8]:
df_train_cleaned.head()


,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
141599,a quote many psychometricians and behavio...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
86692,and right at the time of making such allegati...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
28863,formally not de facto yes . and in the end ...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
14048,religious groups why is the only religion is...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
78005,the doctor suggests it ' s probably something...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0


In [9]:
df_val_cleaned.head()


,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
50446,and redirect the other names to it,0,0,0,0,0,0
81571,sinebot1 please read the above comments .,0,0,0,0,0,0
25983,thank you for your very good answer . i just...,0,0,0,0,0,0
39022,i think we need to ask who is likely to be v...,0,0,0,0,0,0
49431,orangemonster2k1|svrtvdude]] ( vt ) 23 26 3...,0,0,0,0,0,0


In [10]:
model_dict = {}
all_preds = []
for col in y_cols:
    toxic_preds = []
    # Write files to disk as fastText classifier API reads files from disk.
    train_file = '/kaggle/working/final_train.csv'
    df_train_cleaned[[col, "comment_text"]].to_csv(train_file, header=None, index=False, columns=[col, "comment_text"])
#     test_file = '/kaggle/working/final_test.csv'
#     df_val_cleaned[[col, "comment_text"]].to_csv(test_file, header=None, index=False, columns=[col, "comment_text"])
        
    model = train_supervised(input=train_file, label="__class__", lr=1.0, epoch=2, loss='ova', wordNgrams=2, dim=200, thread=2, verbose=100)
    
    for text in tqdm(df_val_cleaned["comment_text"].values):
#         print(text)
        pred = model.predict(text, k = 2)[1][1]
        toxic_preds.append(pred)
        
    all_preds.append(toxic_preds)


Read 9M words
Number of words:  211652
Number of labels: 2
Progress: 100.0% words/sec/thread: 1124394 lr:  0.000000 avg.loss:  0.257604 ETA:   0h 0m 0s


  0%|          | 0/39893 [00:00<?, ?it/s]

Read 9M words
Number of words:  211652
Number of labels: 2
Progress: 100.0% words/sec/thread: 1406979 lr:  0.000000 avg.loss:  0.071532 ETA:   0h 0m 0s


  0%|          | 0/39893 [00:00<?, ?it/s]

Read 9M words
Number of words:  211652
Number of labels: 2
Progress: 100.0% words/sec/thread: 1124426 lr:  0.000000 avg.loss:  0.161671 ETA:   0h 0m 0s


...

  0%|          | 0/39893 [00:00<?, ?it/s]

Read 9M words
Number of words:  211652
Number of labels: 2
Progress: 100.0% words/sec/thread: 1098286 lr:  0.000000 avg.loss:  0.187757 ETA:   0h 0m 0s


  0%|          | 0/39893 [00:00<?, ?it/s]

Read 9M words
Number of words:  211652
Number of labels: 2
Progress: 100.0% words/sec/thread: 1002849 lr:  0.000000 avg.loss:  0.070026 ETA:   0h 0m 0s


  0%|          | 0/39893 [00:00<?, ?it/s]

In [11]:
all_preds_array = np.transpose(np.array(all_preds))


In [12]:
def accuracy(y_test, y_pred):
    aucs = []
    for col in range(y_test.shape[1]):
        aucs.append(roc_auc_score(y_test[:,col],y_pred[:,col]))
    return aucs


In [13]:
df_val_cleaned[y_cols].astype("int").to_numpy()


array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]])

In [14]:
y_val_actuals = df_val_cleaned[y_cols].astype("int").to_numpy()
mean_auc = mean(accuracy(y_val_actuals,all_preds_array))
mean_auc


0.7713376419344882

In [15]:
df_test = pd.merge(test_text, sample_submission, on = "id")

df_test_cleaned = clean_df(df_test, True, True, label_prefix='')

all_test_preds = []

for col in tqdm(y_cols):
    toxic_preds = []
    for text in df_test_cleaned["comment_text"].values:
    #         print(text)
        pred = model.predict(text, k = 2)[1][1]
        toxic_preds.append(pred)
    all_test_preds.append(toxic_preds)
    
all_test_preds_array = np.transpose(np.array(all_test_preds))

df_test[["toxic","severe_toxic","obscene","threat","insult","identity_hate"]] = all_test_preds_array

df_test.drop(["comment_text"], axis = 1, inplace = True)

df_test.to_csv("sample_submission.csv", index = False)


  0%|          | 0/6 [00:00<?, ?it/s]